# Aula prática de classificação: do EDA à avaliação crítica do modelo

**Autor:** Lucas Mascarenhas Almeida  
**GitHub:** [mascalmeida](https://github.com/mascalmeida)  
**E-mail:** lucasmascalmeida@gmail.com  
**LinkedIn:** [lucas-mascarenhas](https://www.linkedin.com/in/lucas-mascarenhas/)  

**Referência principal:** *An Introduction to Statistical Learning with Applications in Python* — ISLP.

---

## Perguntas-guia da aula

1. O que diferencia uma análise exploratória de uma etapa de modelagem?
2. Como um modelo de classificação transforma variáveis de entrada em uma decisão?
3. Como saber se um modelo está aprendendo algo útil ou apenas reproduzindo a classe majoritária?
4. Como nossas decisões de pré-processamento afetam a avaliação final?
5. Como comparar modelos diferentes de forma justa?


## Antes de começar: perguntas de orientação

1. Qual problema queremos resolver?
2. Qual decisão um modelo poderia apoiar?
3. Qual seria a classe positiva neste tipo de problema?
4. Qual erro poderia ser mais caro: falso positivo ou falso negativo?
5. Como podemos transformar um problema real em uma pergunta de classificação?
6. Como podemos organizar um ciclo de trabalho baseado em hipótese, experimento, evidência e diagnóstico?


# Preparação do ambiente

In [ ]:
# No Google Colab, esta biblioteca costuma instalar rapidamente.
# Ela será usada no Ato 3, quando testarmos SMOTE.

!pip -q install imbalanced-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

from imblearn.over_sampling import SMOTE

pd.set_option("display.max_columns", 100)
pd.set_option("display.precision", 4)

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Ato 1 — Classificação e análise exploratória

## 1.1 Debate introdutório: o que é classificação?

### Perguntas para a turma

1. O que é um modelo de classificação?
2. Qual é a diferença entre classificação e regressão?
3. Quais problemas reais podem ser formulados como classificação?
4. O que significa dizer que a variável resposta é categórica?
5. Em uma classificação binária, o que significam as classes `0` e `1`?
6. Em um problema real, errar a classe `0` tem o mesmo custo que errar a classe `1`?
7. Como saber se um problema deve ser tratado como classificação binária, multiclasse ou regressão?


## 1.2 Ingestão dos dados

### Perguntas para debate

1. De onde vêm os dados?
2. Cada linha e cada coluna parecem representar o quê?
3. Que problemas podem aparecer logo na leitura dos dados?


In [ ]:
url = "https://raw.githubusercontent.com/mascalmeida/classification_class_2026/main/data/input_data.csv"

df = pd.read_csv(url)

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## 1.3 Definindo a variável-alvo

### Perguntas para debate

1. Qual coluna parece ser a variável-alvo?
2. Que tipo de variável é a variável-alvo?
3. O que aconteceria se a variável-alvo entrasse também como variável de entrada?


In [ ]:
target = "Fail"

df[target] = df[target].replace({
    True: 1,
    False: 0,
    "True": 1,
    "False": 0,
    "TRUE": 1,
    "FALSE": 0
})

df[target] = df[target].astype(int)

df[target].unique()

## 1.4 Primeira checagem de qualidade dos dados

### Perguntas para debate

1. Quais tipos de dados aparecem em cada coluna?
2. Há valores ausentes?
3. Uma coluna com muitos ou poucos valores únicos exige algum cuidado?


In [ ]:
resumo_colunas = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "percentual_missing": (100 * df.isna().mean()).round(2),
    "n_unique": df.nunique()
})

resumo_colunas

## 1.5 Estatísticas descritivas

### Perguntas para debate

1. Quais variáveis têm maiores e menores escalas?
2. Alguma variável parece ter valores extremos?
3. Por que olhar estatísticas descritivas antes de treinar um modelo?


In [ ]:
df.describe().T

## 1.6 Distribuição da variável-alvo

### Perguntas para debate

1. As classes estão balanceadas?
2. O que aconteceria se um modelo sempre previsse a classe mais frequente?
3. Acurácia seria uma métrica confiável neste caso?


In [ ]:
contagem_classes = df[target].value_counts().sort_index()

resumo_classes = pd.DataFrame({
    "classe": contagem_classes.index,
    "n_observacoes": contagem_classes.values,
    "percentual": (100 * contagem_classes.values / contagem_classes.sum()).round(2)
})

resumo_classes

In [ ]:
contagem_classes.plot(kind="bar", figsize=(6, 4))

plt.title("Distribuição da variável-alvo")
plt.xlabel("Classe")
plt.ylabel("Número de observações")
plt.xticks(rotation=0)
plt.show()

acuracia_classe_majoritaria = contagem_classes.max() / contagem_classes.sum()

print("Acurácia de um modelo que sempre prevê a classe majoritária:")
print(f"{acuracia_classe_majoritaria:.2%}")

## 1.7 Escolhendo as variáveis de entrada

### Perguntas para debate

1. Quais colunas devem entrar como variáveis explicativas?
2. Existe alguma coluna que possa gerar vazamento de informação?
3. Como a escolha das variáveis de entrada afeta a interpretação do modelo?


In [ ]:
feature_cols = [
    "Temperature",
    "Pressure",
    "VibrationX",
    "VibrationY",
    "VibrationZ",
    "Frequency",
    "Preset_1",
    "Preset_2"
]

df[feature_cols + [target]].head()

## 1.8 Histogramas das variáveis de entrada

### Perguntas para debate

1. Alguma variável parece ter valores extremos?
2. As variáveis parecem estar em escalas parecidas ou muito diferentes?
3. O que esses gráficos sugerem sobre pré-processamento?


In [ ]:
df[feature_cols].hist(figsize=(14, 10), bins=30)

plt.suptitle("Histogramas das variáveis de entrada", y=1.02)
plt.tight_layout()
plt.show()

## 1.9 Comparando médias por classe

### Perguntas para debate

1. Quais variáveis parecem ter médias diferentes entre as classes?
2. Uma diferença de médias garante que a variável será útil no modelo?
3. O tamanho da classe minoritária pode afetar essa comparação?


In [ ]:
medias_por_classe = df.groupby(target)[feature_cols].mean().T

medias_por_classe.columns = ["media_classe_0", "media_classe_1"]
medias_por_classe["diferenca_1_menos_0"] = (
    medias_por_classe["media_classe_1"] - medias_por_classe["media_classe_0"]
)

medias_por_classe.sort_values("diferenca_1_menos_0", ascending=False)

## 1.10 Correlação entre variáveis

### Perguntas para debate

1. O que significa duas variáveis terem correlação alta?
2. Uma variável pouco correlacionada com o alvo deve ser descartada automaticamente?
3. Modelos lineares e modelos baseados em árvores lidam da mesma forma com correlação?


In [ ]:
corr = df[feature_cols + [target]].corr(method="pearson")

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Matriz de correlação (Pearson)")
plt.show()

## 1.11 Uma olhada em variáveis de configuração

### Perguntas para debate

1. Quais variáveis parecem ter poucos valores únicos?
2. A taxa da classe positiva muda entre os valores dessas variáveis?
3. Uma diferença visual nas taxas é suficiente para concluir alguma coisa?


In [ ]:
resumo_preset_1 = (
    df.groupby("Preset_1")[target]
    .agg(n_observacoes="size", positivos="sum", taxa_positiva="mean")
    .reset_index()
)

resumo_preset_1["taxa_positiva_percentual"] = (
    100 * resumo_preset_1["taxa_positiva"]
).round(2)

resumo_preset_1

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(resumo_preset_1["Preset_1"].astype(str), resumo_preset_1["taxa_positiva"])

plt.title("Taxa da classe positiva por Preset_1")
plt.xlabel("Preset_1")
plt.ylabel("Taxa da classe positiva")
plt.show()

In [ ]:
resumo_preset_2 = (
    df.groupby("Preset_2")[target]
    .agg(n_observacoes="size", positivos="sum", taxa_positiva="mean")
    .reset_index()
)

resumo_preset_2["taxa_positiva_percentual"] = (
    100 * resumo_preset_2["taxa_positiva"]
).round(2)

resumo_preset_2

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(resumo_preset_2["Preset_2"].astype(str), resumo_preset_2["taxa_positiva"])

plt.title("Taxa da classe positiva por Preset_2")
plt.xlabel("Preset_2")
plt.ylabel("Taxa da classe positiva")
plt.show()

# Ato 2 — Primeiro modelo simples

## 2.1 Debate: normalização, treino/teste, desbalanceamento e pré-processamento

### Perguntas para a turma

1. Por que dividimos dados em treino e teste?
2. O que o conjunto de treino deve simular?
3. O que o conjunto de teste deve simular?
4. Por que normalizamos algumas variáveis antes de treinar determinados modelos?
5. O que pode dar errado se normalizarmos usando o dataset inteiro?
6. O que são dados desbalanceados?
7. Por que um bom EDA ajuda antes da modelagem?


## 2.2 Separando `X` e `y`

### Perguntas para debate

1. O que deve entrar em `X`?
2. O que deve entrar em `y`?
3. O que aconteceria se a variável-alvo fosse incluída em `X`?


## 2.3 Divisão aleatória entre treino e teste

### Perguntas para debate

1. Por que não avaliamos o modelo nos mesmos dados usados para treino?
2. O que esperamos preservar ao usar `stratify=y`?
3. Por que pode ser útil começar pelo modelo mais simples possível?


## 2.4 Normalização

### Perguntas para debate

1. Por que algumas variáveis precisam ser colocadas em escalas comparáveis?
2. Em qual conjunto devemos dar `fit` no normalizador?
3. Por que não devemos dar `fit` no conjunto de teste?


## 2.5 Modelo 1 — Regressão logística com dados desbalanceados

### Perguntas para debate

1. Por que um modelo chamado regressão logística pode ser usado em classificação?
2. O que a probabilidade prevista pelo modelo representa?
3. O limiar de `0.50` é sempre adequado?


## 2.6 Avaliação do primeiro modelo

### Perguntas para debate

1. Por que a acurácia pode não contar a história inteira?
2. O que precision e recall podem revelar que a acurácia esconde?
3. O que a curva ROC pode acrescentar à avaliação?


# Ato 3 — Interpretando, diagnosticando e melhorando

## 3.0 Debate interpretativo: o que significam as métricas?

### Perguntas para a turma

1. O que representa cada célula da matriz de confusão?
2. Qual é a diferença entre falso positivo e falso negativo?
3. O que precision responde sobre as previsões positivas?
4. O que recall responde sobre os casos positivos reais?
5. O que o f1-score tenta equilibrar?
6. Como a curva ROC ajuda a comparar limiares de decisão?
7. Dado o primeiro resultado, quais hipóteses podemos testar para melhorar?


## 3.1 Balanceamento de classe com SMOTE

### Perguntas para debate

1. Por que não balanceamos o conjunto de teste?
2. Qual é a diferença entre usar SMOTE e apenas alterar o peso das classes?
3. Como saber se o balanceamento ajudou ou apenas mudou o tipo de erro?


## 3.2 Modelo 2 — Regressão logística com SMOTE

### Perguntas para debate

1. O que muda em relação à regressão logística anterior?
2. O balanceamento do treino muda a distribuição do teste?
3. Como decidir se o SMOTE ajudou?


## 3.3 Modelo 3 — Random Forest com `class_weight` e dados desbalanceados

### Perguntas para debate

1. O que muda quando saímos de regressão logística para Random Forest?
2. Por que testar `class_weight` em dados desbalanceados?
3. Um modelo mais complexo sempre melhora o resultado?


## 3.4 Importância de variáveis na Random Forest

### Perguntas para debate

1. Quais variáveis aparecem como mais importantes?
2. Importância de variável prova causalidade?
3. A importância pode mudar se mudarmos o split ou os hiperparâmetros?


## 3.5 Modelo 4 — Random Forest com SMOTE

### Perguntas para debate

1. O que muda em relação à Random Forest com `class_weight`?
2. O SMOTE afeta modelos lineares e modelos baseados em árvores da mesma forma?
3. Que resultado indicaria que o modelo está gerando positivos demais?


## 3.6 Comparando os quatro modelos

### Perguntas para debate

1. Qual modelo teve maior recall, precision e f1-score?
2. Algum modelo parece bom em uma métrica, mas ruim em outra?
3. Qual modelo você escolheria para investigar mais? Por quê?


## 3.7 Discussão: o que tentar depois?

### Perguntas para próximos experimentos

1. O que aconteceria se ajustássemos o limiar de decisão em vez de usar sempre `0.50`?
2. Como poderíamos testar se uma divisão temporal muda a avaliação do modelo?
3. Quando faria sentido comparar classificação supervisionada com detecção de anomalias?
4. Eu quero prever não apenas o momento da falha, mas operações perto do limite operacional da máquina, isso permitiria aumentar a quantidade de observações de "fail".


# Fechamento — Como pensar como cientista de dados

## Perguntas finais para a turma

1. Qual foi o principal diagnóstico depois do primeiro modelo?
2. Qual hipótese o SMOTE testou?
3. Qual hipótese o `class_weight` testou?
4. Que evidência faria você escolher um dos quatro modelos?
5. Como poderíamos testar, em uma próxima aula, a hipótese de que o split temporal muda a avaliação?
6. Que outro experimento futuro você testaria: ajuste de limiar, novas variáveis, validação cruzada, outro modelo ou detecção de anomalias?
7. Como você explicaria o ciclo diagnóstico → hipótese → experimento → evidência → novo diagnóstico?
